# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** · Turismo e Hotelaria

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu as 119.390 reservas
brutas de hotéis urbanos e resorts. Este notebook aplica somente ajustes documentados,
preserva as observações e cria variáveis úteis para analisar a estadia.

A regra desta etapa é:

> **Nenhum dado é alterado em silêncio.** Cada ajuste registra o motivo, a quantidade de
> valores afetados e o arquivo de saída.

In [ ]:
# Importações
import pandas as pd

In [ ]:
# Carregando a base bruta de hotelaria
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQUIVO_BRUTO = RAIZ / "data" / "raw" / "hotel_bookings.csv"
ARQUIVO_TRATADO = RAIZ / "data" / "processed" / "hotel_bookings_tratado.csv"
ARQUIVO_AUDITORIA = RAIZ / "reports" / "tables" / "ajustes-dados.csv"
ARQUIVO_TRATADO.parent.mkdir(parents=True, exist_ok=True)
ARQUIVO_AUDITORIA.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    ARQUIVO_BRUTO,
    low_memory=False,
    dtype={"agent": "string", "company": "string", "country": "string"},
)

linhas_iniciais = len(df)
auditoria = []


def registrar(coluna, regra, quantidade):
    auditoria.append(
        {
            "coluna": coluna,
            "regra": regra,
            "quantidade_afetada": int(quantidade),
        }
    )


print(f"{linhas_iniciais:,} reservas carregadas; {df.shape[1]} colunas originais")

In [ ]:
# Ajustes documentados para a base de reservas hoteleiras

# A ausência de children representa a ausência de crianças informadas na reserva.
quantidade = int(df["children"].isna().sum())
df["children"] = df["children"].fillna(0)
registrar("children", "Ausente convertido para 0 crianças", quantidade)

# Ausências de agência, empresa e país permanecem informativas como categorias explícitas.
for coluna, rotulo in [
    ("agent", "Sem agência"),
    ("company", "Sem empresa"),
    ("country", "País não informado"),
]:
    quantidade = int(df[coluna].isna().sum())
    df[coluna] = df[coluna].fillna(rotulo)
    registrar(coluna, f"Ausente convertido para categoria: {rotulo}", quantidade)

# Converte a data de atualização do status para o tipo temporal.
df["reservation_status_date"] = pd.to_datetime(df["reservation_status_date"], errors="coerce")
registrar(
    "reservation_status_date",
    "Conversão de texto para data; valores inválidos viram nulos",
    int(df["reservation_status_date"].isna().sum()),
)

# Recompõe a data prevista de chegada a partir dos campos do calendário.
meses = pd.to_datetime(df["arrival_date_month"], format="%B", errors="coerce").dt.month
df["arrival_date"] = pd.to_datetime(
    {
        "year": df["arrival_date_year"],
        "month": meses,
        "day": df["arrival_date_day_of_month"],
    },
    errors="coerce",
)
registrar("arrival_date", "Data de chegada recomposta de ano, mês e dia", len(df))

# Variáveis derivadas de estadia e capacidade da reserva.
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df[["adults", "children", "babies"]].sum(axis=1)
df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype("int8")
df["has_special_requests"] = (df["total_of_special_requests"] > 0).astype("int8")

registrar("total_nights", "Soma de noites de fim de semana e durante a semana", len(df))
registrar("total_guests", "Soma de adultos, crianças e bebês", len(df))
registrar(
    "room_changed", "Quarto atribuído diferente do quarto reservado", int(df["room_changed"].sum())
)
registrar(
    "has_special_requests",
    "Reserva com pelo menos uma solicitação especial",
    int(df["has_special_requests"].sum()),
)

# A base não tem uma chave única: duplicatas exatas são preservadas.
registrar(
    "linhas",
    "Duplicatas exatas preservadas por não haver chave de reserva",
    int(df.duplicated().sum()),
)

# Status final e sua data são mantidos para análise, mas não devem prever cancelamento.
registrar(
    "reservation_status",
    "Mantido como desfecho posterior; excluir de modelos de previsão",
    0,
)
registrar(
    "reservation_status_date",
    "Mantido como informação posterior; excluir de modelos de previsão",
    0,
)

nulos_restantes = df.isna().sum()
print("Nulos restantes por coluna:")
print(nulos_restantes[nulos_restantes > 0])
print(f"Linhas preservadas: {len(df):,} de {linhas_iniciais:,}")

assert len(df) == linhas_iniciais
assert int(df.isna().sum().sum()) == 0

# A data é gravada em formato ISO para manter o CSV portátil.
df["reservation_status_date"] = df["reservation_status_date"].dt.strftime("%Y-%m-%d")
df["arrival_date"] = df["arrival_date"].dt.strftime("%Y-%m-%d")
df.to_csv(ARQUIVO_TRATADO, sep=";", index=False, encoding="utf-8")

pd.DataFrame(auditoria).to_csv(
    ARQUIVO_AUDITORIA,
    sep=";",
    index=False,
    encoding="utf-8",
)

print(f"Base tratada salva em: {ARQUIVO_TRATADO}")
print(f"Auditoria salva em: {ARQUIVO_AUDITORIA}")